# De l'ACP aux auto-encodeurs profonds avec Keras

Un auto-encodeur comprime son entrée en un code de petite taille (l'encodeur), puis la reconstruit depuis ce code (le décodeur) : sa cible est son entrée elle-même. Ce notebook en entraîne plusieurs sur MNIST (60 000 images d'entraînement et 10 000 de test, des chiffres manuscrits 28 × 28 en niveaux de gris), du plus simple au plus profond :

1. Un auto-encodeur linéaire
2. L'auto-encodeur linéaire face à l'ACP
3. Un auto-encodeur profond
4. L'espace latent
5. La détection d'anomalies
6. La taille du code
7. Pour aller plus loin : un auto-encodeur épars

Chaque partie réutilise les modèles des précédentes ; la dernière est facultative. Pas besoin de GPU : le notebook tourne en une vingtaine de minutes sur un processeur, dont la moitié pour la partie 6, qui entraîne six auto-encodeurs.

In [ ]:
import keras
import matplotlib.pyplot as plt
import numpy as np
from keras import layers
from sklearn.decomposition import PCA

keras.utils.set_random_seed(0)

(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()
input_dim = 28 * 28
# Pixels dans [0, 1], comme la sortie d'une sigmoïde : ni centrés, ni réduits
X_train = X_train.reshape(-1, input_dim) / 255
X_test = X_test.reshape(-1, input_dim) / 255
print(X_train.shape, X_test.shape)


def show_images(*rows: np.ndarray, titles: list | np.ndarray | None = None) -> None:
  # Affiche des rangées d'images 28 × 28, une rangée par tableau
  n = len(rows[0])
  _, axes = plt.subplots(
    len(rows), n, figsize=(n * 1.2, len(rows) * 1.3), squeeze=False
  )
  for row, images in zip(axes, rows):
    for ax, image in zip(row, images):
      ax.imshow(np.reshape(image, (28, 28)), cmap="gray_r")
      ax.axis("off")
  for ax, title in zip(axes[0], [] if titles is None else titles):
    ax.set_title(title)
  plt.show()


show_images(X_train[:10], titles=y_train[:10])

## 1. Un auto-encodeur linéaire

Sans fonction d'activation, l'encodeur et le décodeur ne sont que des applications linéaires (plus un biais). Construisez, comme sur la slide « Auto-encodeur en Keras » mais sans activation :

- `encoder` : une couche `Dense` de `encoding_dim = 20` unités, sur des entrées de taille `input_dim` ;
- `decoder` : une couche `Dense` de `input_dim` unités, sur des codes de taille `encoding_dim` ;
- `autoencoder`, qui enchaîne les deux, compilé avec l'optimiseur Adam et la perte `"mse"`.

La cellule après la solution l'entraîne.

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `keras.Input(shape=(input_dim,))` en tête de chaque `keras.Sequential`, qui fixe la taille de ses entrées
- `layers.Dense(n)` sans argument `activation` : une couche linéaire
- `keras.Sequential([encoder, decoder], name="autoencoder")` : un modèle peut servir de couche à un autre, et `encoder` et `decoder` restent utilisables seuls
- `autoencoder.summary()` doit afficher 15 700 paramètres pour l'encodeur (784 × 20 poids + 20 biais) et 16 464 pour le décodeur

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
encoding_dim = 20
encoder = keras.Sequential(
  [keras.Input(shape=(input_dim,)), layers.Dense(...)], name="encoder"
)
decoder = keras.Sequential([..., ...], name="decoder")
autoencoder = keras.Sequential([..., ...], name="autoencoder")
autoencoder.compile(optimizer=..., loss=...)
autoencoder.summary()
```

### Solution

In [ ]:
encoding_dim = 20
encoder = keras.Sequential(
  [keras.Input(shape=(input_dim,)), layers.Dense(encoding_dim)], name="encoder"
)
decoder = keras.Sequential(
  [keras.Input(shape=(encoding_dim,)), layers.Dense(input_dim)], name="decoder"
)
autoencoder = keras.Sequential([encoder, decoder], name="autoencoder")
autoencoder.compile(optimizer="adam", loss="mse")
autoencoder.summary()

### Entraînement

La cible est l'entrée : `fit(X_train, X_train)`. La perte de validation se calcule sur les images de test.

In [ ]:
autoencoder.fit(
  X_train,
  X_train,
  epochs=10,
  batch_size=256,
  validation_data=(X_test, X_test),
  verbose=2,
)
linear_decoder = decoder  # gardé pour la partie 2
X_test_linear = autoencoder.predict(X_test, verbose=0)
show_images(X_test[:10], X_test_linear[:10], titles=y_test[:10])

## 2. L'auto-encodeur linéaire face à l'ACP

La slide « Auto-encodeur linéaire et ACP » l'affirme : entraîné avec la MSE, un auto-encodeur linéaire à code de taille k retrouve le sous-espace des k premières composantes de l'ACP, mais pas ses axes. Vérifiez-le :

- ajustez une `PCA` à `encoding_dim` composantes sur `X_train`, reconstruisez `X_test` avec elle, et comparez l'erreur quadratique moyenne de sa reconstruction à celle de l'auto-encodeur (`X_test_linear`, calculé par la cellule précédente) ;
- les 20 directions du décodeur sont les lignes de son noyau `W` (20 × 784) : mesurez, pour chacune, la part de sa norme qui tombe dans le sous-espace de l'ACP ;
- mesurez enfin le plus grand cosinus, en valeur absolue, entre deux directions du décodeur, puis entre deux composantes de l'ACP.

Les deux erreurs sont-elles proches ? Les directions du décodeur sont-elles dans le sous-espace de l'ACP ? Sont-elles orthogonales, comme les composantes ?

**Documentation utile**

- [`sklearn.decomposition.PCA`](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html)
- [`Layer.get_weights`](https://keras.io/api/layers/base_layer/#getweights-method)
- [`numpy.linalg.norm`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.norm.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `pca = PCA(encoding_dim).fit(X_train)`, puis `pca.inverse_transform(pca.transform(X_test))` : la reconstruction par l'ACP ; l'erreur : `np.mean((X_test - X_rec) ** 2)`
- `W = linear_decoder.layers[0].get_weights()[0]` : le noyau de l'unique couche `Dense` du décodeur, de forme (20, 784), une direction par ligne (`get_weights()[1]` est le biais)
- les lignes de `pca.components_` (20 × 784) sont orthonormées : `W @ pca.components_.T` donne les coordonnées de chaque direction dans la base de l'ACP ; la norme de chaque ligne de ce produit (`np.linalg.norm(..., axis=1)`), divisée par celle de la ligne de `W`, est la part cherchée (1 : la direction est entièrement dans le sous-espace)
- les cosinus : normalisez les lignes, `V / np.linalg.norm(V, axis=1, keepdims=True)`, puis `V @ V.T` ; hors de la diagonale (`~np.eye(20, dtype=bool)`), 0 pour deux directions orthogonales

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
pca = PCA(...).fit(...)
X_test_pca = ...
pca_error = np.mean((X_test - X_test_pca) ** 2)
linear_error = ...
print(f"Erreur : ACP {pca_error:.4f}, AE linéaire {linear_error:.4f}")

W = linear_decoder.layers[0].get_weights()[0]
share = ...
print(f"Part dans le sous-espace de l'ACP : {share.min():.3f} à {share.max():.3f}")


def max_cosine(V: np.ndarray) -> float:
  V = ...
  cosines = ...
  return np.abs(cosines[~np.eye(len(V), dtype=bool)]).max()


print(f"Plus grand cosinus : décodeur {max_cosine(W):.2f}, ACP {max_cosine(...):.2f}")
```

### Solution

In [ ]:
pca = PCA(encoding_dim).fit(X_train)
X_test_pca = pca.inverse_transform(pca.transform(X_test))
pca_error = np.mean((X_test - X_test_pca) ** 2)
linear_error = np.mean((X_test - X_test_linear) ** 2)
print(f"Erreur : ACP {pca_error:.4f}, AE linéaire {linear_error:.4f}")

W = linear_decoder.layers[0].get_weights()[0]  # (20, 784)
share = np.linalg.norm(W @ pca.components_.T, axis=1) / np.linalg.norm(W, axis=1)
print(f"Part dans le sous-espace de l'ACP : de {share.min():.3f} à {share.max():.3f}")


def max_cosine(V: np.ndarray) -> float:
  V = V / np.linalg.norm(V, axis=1, keepdims=True)
  cosines = V @ V.T
  return np.abs(cosines[~np.eye(len(V), dtype=bool)]).max()


print(
  f"Plus grand cosinus : décodeur {max_cosine(W):.2f}, ACP {max_cosine(pca.components_):.2f}"
)

Les deux erreurs sont presque égales : l'auto-encodeur linéaire fait aussi bien que l'ACP, mais pas mieux, puisque l'ACP donne la meilleure reconstruction linéaire possible à 20 dimensions. Ses 20 directions sont presque entièrement dans le sous-espace de l'ACP (à plus de 95 %) : il a trouvé le même sous-espace, aux imprécisions de l'entraînement près. Mais elles ne sont pas orthogonales (le plus grand cosinus vaut environ 0,2, contre 0 pour l'ACP), ni triées par variance expliquée : rien dans la perte ne l'exige. L'ACP donne cette base en plus, et sans entraînement.

## 3. Un auto-encodeur profond

Ajoutez une couche cachée non linéaire de part et d'autre du code, comme sur la slide « Auto-encodeur en Keras » :

- `encoder` : une `Dense` de 128 unités, puis une de `encoding_dim` unités, toutes deux avec l'activation ReLU ;
- `decoder` : une `Dense` de 128 unités (ReLU), puis la couche de sortie, de `input_dim` unités ;
- `autoencoder`, qui enchaîne les deux.

Choisissez l'activation de la couche de sortie et la perte, sachant que les pixels sont dans [0, 1], et justifiez votre choix en une ligne de commentaire. La cellule après la solution l'entraîne, puis compare son erreur à celles de l'ACP et de l'auto-encodeur linéaire.

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `layers.Dense(128, activation="relu")` : une couche cachée
- la sortie : `activation="sigmoid"`, qui rend des valeurs dans ]0, 1[, comme les pixels ; une sortie linéaire peut rendre des pixels négatifs ou plus grands que 1
- la perte : `"binary_crossentropy"`, qui traite chaque pixel comme une probabilité (la cible peut être entre 0 et 1, pas seulement 0 ou 1), ou `"mse"`, qui fonctionne aussi
- `autoencoder.summary()` : 103 060 paramètres pour l'encodeur, 103 824 pour le décodeur

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
encoder = keras.Sequential(
  [
    keras.Input(shape=(input_dim,)),
    layers.Dense(128, activation="relu"),
    ...,
  ],
  name="encoder",
)
decoder = keras.Sequential(
  [keras.Input(shape=(encoding_dim,)), ..., layers.Dense(input_dim, activation=...)],
  name="decoder",
)
autoencoder = keras.Sequential([encoder, decoder], name="autoencoder")
autoencoder.compile(optimizer="adam", loss=...)  # parce que ...
autoencoder.summary()
```

### Solution

In [ ]:
encoder = keras.Sequential(
  [
    keras.Input(shape=(input_dim,)),
    layers.Dense(128, activation="relu"),
    layers.Dense(encoding_dim, activation="relu"),
  ],
  name="encoder",
)
decoder = keras.Sequential(
  [
    keras.Input(shape=(encoding_dim,)),
    layers.Dense(128, activation="relu"),
    # Sortie dans ]0, 1[, comme les pixels
    layers.Dense(input_dim, activation="sigmoid"),
  ],
  name="decoder",
)
autoencoder = keras.Sequential([encoder, decoder], name="autoencoder")
# Chaque pixel cible, dans [0, 1], est traité comme une probabilité
autoencoder.compile(optimizer="adam", loss="binary_crossentropy")
autoencoder.summary()

### Entraînement et comparaison

In [ ]:
autoencoder.fit(
  X_train,
  X_train,
  epochs=20,
  batch_size=256,
  validation_data=(X_test, X_test),
  verbose=2,
)
X_test_pred = autoencoder.predict(X_test, verbose=0)
deep_error = np.mean((X_test - X_test_pred) ** 2)
print(
  f"Erreur quadratique moyenne : ACP {pca_error:.4f}, "
  f"AE linéaire {linear_error:.4f}, AE profond {deep_error:.4f}"
)
# En haut les originaux, puis l'AE linéaire, puis l'AE profond
show_images(X_test[:10], X_test_linear[:10], X_test_pred[:10], titles=y_test[:10])

Avec le même code de 20 nombres, l'auto-encodeur profond reconstruit nettement mieux que l'ACP et que l'auto-encodeur linéaire : ses couches non linéaires suivent des surfaces courbes, hors de portée d'un sous-espace.

## 4. L'espace latent

L'encodeur seul transforme chaque image de test en son code, un point d'un espace à `encoding_dim` dimensions : l'espace latent.

In [ ]:
codes = encoder.predict(X_test, verbose=0)
print(codes.shape)

### Où vivent les codes ?

Le décodeur seul pourrait-il fabriquer de nouveaux chiffres à partir de codes tirés au hasard ? Regardez d'abord où vivent les vrais codes, puis essayez :

- pour chacune des 20 dimensions, affichez le minimum, la moyenne et le maximum des `codes`, et comptez les dimensions qui valent 0 pour toutes les images ;
- décodez 10 codes de bruit blanc, tirés uniformément dans [0, 1] sur chaque dimension, puis 10 codes tirés uniformément entre le minimum et le maximum de chaque dimension, et affichez les deux rangées d'images avec `show_images`.

Les codes ressemblent-ils à du bruit uniforme dans [0, 1] ? Les images décodées ressemblent-elles à des chiffres ? Pourquoi ?

**Documentation utile**

- [`numpy.random.Generator.uniform`](https://numpy.org/doc/stable/reference/random/generated/numpy.random.Generator.uniform.html)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `codes.min(axis=0)`, `codes.mean(axis=0)`, `codes.max(axis=0)` : une valeur par dimension ; `np.round(..., 1)` pour les afficher
- une dimension qui vaut 0 pour toutes les images : `codes.max(axis=0) == 0` (la ReLU du code ne rend jamais moins de 0), puis `.sum()`
- `rng = np.random.default_rng(0)`, puis `rng.uniform(0, 1, size=(10, encoding_dim))` ; `rng.uniform(low, high, size=(10, encoding_dim))` accepte aussi des tableaux `low` et `high`, une valeur par dimension
- `decoder.predict(noise, verbose=0)` rend 10 images de 784 pixels, que `show_images` affiche

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
low, high = ..., ...
print("Min :", np.round(low, 1))
print("Moyenne :", ...)
print("Max :", ...)
print("Dimensions mortes :", int(...))

rng = np.random.default_rng(0)
white_noise = rng.uniform(...)
in_range = rng.uniform(...)
show_images(decoder.predict(white_noise, verbose=0), decoder.predict(..., verbose=0))
```

#### Solution

In [ ]:
low, high = codes.min(axis=0), codes.max(axis=0)
print("Min :", np.round(low, 1))
print("Moyenne :", np.round(codes.mean(axis=0), 1))
print("Max :", np.round(high, 1))
print("Dimensions mortes :", int((high == 0).sum()))

rng = np.random.default_rng(0)
white_noise = rng.uniform(0, 1, size=(10, encoding_dim))
in_range = rng.uniform(low, high, size=(10, encoding_dim))
# En haut le bruit blanc, en bas les codes tirés dans leur plage
show_images(
  decoder.predict(white_noise, verbose=0), decoder.predict(in_range, verbose=0)
)

Les codes ne ressemblent pas à du bruit uniforme dans [0, 1] : la ReLU les rend positifs ou nuls, leur plage change d'une dimension à l'autre et va jusqu'à plusieurs dizaines, et une dimension ou plus peut être morte, nulle pour toutes les images. Le bruit blanc tombe donc dans un coin de l'espace latent, près de l'origine, que le décodeur n'a presque jamais vu : il rend des taches qui se ressemblent toutes. Tirés dans leur plage, les codes ne donnent pas mieux : les vrais codes n'occupent qu'une petite partie de ce pavé, car leurs dimensions sont liées entre elles. Un auto-encodeur compresse, mais ne génère pas : son décodeur ne sait décoder que les codes que produit l'encodeur. L'auto-encodeur variationnel, dans le notebook suivant, impose aux codes une loi connue, $\mathcal{N}(0, I)$, dans laquelle on peut tirer.

### Les centroïdes des chiffres

Les codes ne sont pas dispersés au hasard : ceux d'un même chiffre se regroupent. Le code moyen de chaque chiffre, son centroïde, décodé, en donne une image type.

In [ ]:
centroids = np.stack([codes[y_test == digit].mean(axis=0) for digit in range(10)])
show_images(decoder.predict(centroids, verbose=0), titles=list(range(10)))

### Interpoler entre deux centroïdes

Les codes régulièrement espacés sur le segment qui relie deux centroïdes, une fois décodés, montrent comment le décodeur passe d'un chiffre à l'autre. Essayez d'autres paires.

In [ ]:
def latent_walk(start: int, end: int, n: int = 12) -> None:
  # n codes régulièrement espacés, du centroïde de start à celui de end
  walk = np.linspace(centroids[start], centroids[end], n)
  show_images(decoder.predict(walk, verbose=0))


latent_walk(7, 1)
latent_walk(3, 8)

## 5. La détection d'anomalies

Un auto-encodeur reconstruit bien ce qui ressemble à ses exemples d'entraînement, et mal le reste (slide « Détection d'anomalies par erreur de reconstruction »). La cellule suivante reconstruit aussi les images d'entraînement, dans `X_train_pred` ; celles de test sont dans `X_test_pred`. Puis :

- calculez l'erreur quadratique moyenne de reconstruction de chaque image, dans `train_errors` et `test_errors` ;
- fixez le seuil au quantile à 99 % de `train_errors`, et comptez les images de test qui le dépassent ;
- affichez les 10 images de test les plus mal reconstruites au-dessus de leur reconstruction, avec `show_images`, puis les 10 mieux reconstruites.

Les pires sont-elles des chiffres mal écrits, ou des chiffres d'une classe précise ? Et les meilleures ?

**Documentation utile**

- [`numpy.quantile`](https://numpy.org/doc/stable/reference/generated/numpy.quantile.html)
- [`numpy.argsort`](https://numpy.org/doc/stable/reference/generated/numpy.argsort.html)

In [ ]:
X_train_pred = autoencoder.predict(X_train, verbose=0)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `np.mean((X_train - X_train_pred) ** 2, axis=1)` : une erreur par image (la moyenne sur ses 784 pixels)
- `np.quantile(train_errors, 0.99)` : la valeur que dépassent 1 % des erreurs d'entraînement ; `(test_errors > threshold).sum()` compte les images de test au-delà, une centaine si le test ressemble à l'entraînement
- `np.argsort(test_errors)` : les indices des images, de la plus petite erreur à la plus grande ; `[::-1]` inverse l'ordre
- `show_images(X_test[worst], X_test_pred[worst], titles=y_test[worst])` : les originaux au-dessus de leur reconstruction

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
train_errors = ...
test_errors = ...
threshold = ...
print(f"Seuil : {threshold:.4f}, {...} images de test au-delà")

order = np.argsort(test_errors)
worst, best = order[::-1][:10], ...
show_images(X_test[worst], X_test_pred[worst], titles=y_test[worst])
show_images(...)
```

### Solution

In [ ]:
train_errors = np.mean((X_train - X_train_pred) ** 2, axis=1)
test_errors = np.mean((X_test - X_test_pred) ** 2, axis=1)
threshold = np.quantile(train_errors, 0.99)
print(
  f"Seuil : {threshold:.4f}, {(test_errors > threshold).sum()} images de test au-delà"
)

order = np.argsort(test_errors)
worst, best = order[::-1][:10], order[:10]
print("Les pires reconstructions")
show_images(X_test[worst], X_test_pred[worst], titles=y_test[worst])
print("Les meilleures")
show_images(X_test[best], X_test_pred[best], titles=y_test[best])

Environ 1 % des images de test dépassent le seuil, comme attendu quand le test ressemble à l'entraînement. Les pires reconstructions sont des chiffres de forme rare, surtout des 8 et des 6 aux boucles inhabituelles : ils s'éloignent de ce que l'auto-encodeur a vu, et sa reconstruction les ramène vers un chiffre plus courant. Les meilleures sont presque toutes des 1 : peu de pixels allumés, une forme simple, donc peu d'erreur. L'erreur dépend donc aussi de la classe : avec un seul seuil, un chiffre compliqué est plus souvent signalé qu'un 1.

## 6. La taille du code

Un code plus grand reconstruit mieux, mais compresse moins. Mesurez ce compromis :

- écrivez `build_autoencoder(encoding_dim)`, qui construit et compile l'auto-encodeur profond de la partie 3 pour une taille de code donnée ;
- entraînez-en un pour chaque taille de code de 2, 4, 8, 16, 32 et 64, pendant 10 epochs chacun, avec le même `fit` que dans la partie 3 (et `verbose=0`) ;
- gardez chaque modèle entraîné dans un dictionnaire `autoencoders`, et sa dernière perte de validation dans `val_losses`, indexés par la taille du code ;
- tracez la perte de validation en fonction de la taille du code, avec un axe des abscisses en échelle logarithmique de base 2.

Quelle taille de code choisiriez-vous ? Quel taux de compression (784 / taille du code) obtenez-vous alors ?

**Documentation utile**

- [`keras.callbacks.History`](https://keras.io/api/models/model_training_apis/#fit-method)
- [`matplotlib.pyplot.xscale`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.xscale.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- le corps de `build_autoencoder` est la solution de la partie 3, avec `encoding_dim` en paramètre, suivie de `return autoencoder`
- `history = autoencoder.fit(...)` : `history.history["val_loss"]` est la liste des pertes de validation, une par epoch, et `[-1]` la dernière
- la boucle prend quelques minutes : `print` la perte de chaque taille au fil de l'eau
- `plt.plot(list(val_losses), list(val_losses.values()), marker="o")`, puis `plt.xscale("log", base=2)`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def build_autoencoder(encoding_dim: int) -> keras.Model:
  encoder = ...
  decoder = ...
  autoencoder = keras.Sequential([encoder, decoder])
  autoencoder.compile(...)
  return autoencoder


autoencoders, val_losses = {}, {}
for size in [2, 4, 8, 16, 32, 64]:
  autoencoders[size] = build_autoencoder(size)
  history = autoencoders[size].fit(...)
  val_losses[size] = ...
  print(f"Code de taille {size} : {val_losses[size]:.4f}")

plt.plot(...)
plt.xscale(...)
plt.xlabel("Taille du code")
plt.ylabel("Perte de validation")
plt.show()
```

### Solution

In [ ]:
def build_autoencoder(encoding_dim: int) -> keras.Model:
  encoder = keras.Sequential(
    [
      keras.Input(shape=(input_dim,)),
      layers.Dense(128, activation="relu"),
      layers.Dense(encoding_dim, activation="relu"),
    ],
    name="encoder",
  )
  decoder = keras.Sequential(
    [
      keras.Input(shape=(encoding_dim,)),
      layers.Dense(128, activation="relu"),
      layers.Dense(input_dim, activation="sigmoid"),
    ],
    name="decoder",
  )
  autoencoder = keras.Sequential([encoder, decoder], name="autoencoder")
  autoencoder.compile(optimizer="adam", loss="binary_crossentropy")
  return autoencoder


autoencoders, val_losses = {}, {}
for size in [2, 4, 8, 16, 32, 64]:
  autoencoders[size] = build_autoencoder(size)
  history = autoencoders[size].fit(
    X_train,
    X_train,
    epochs=10,
    batch_size=256,
    validation_data=(X_test, X_test),
    verbose=0,
  )
  val_losses[size] = history.history["val_loss"][-1]
  print(
    f"Code de taille {size} (compression {input_dim / size:.0f}x) : "
    f"perte de validation {val_losses[size]:.4f}"
  )

plt.plot(list(val_losses), list(val_losses.values()), marker="o")
plt.xscale("log", base=2)
plt.xlabel("Taille du code")
plt.ylabel("Perte de validation")
plt.show()

La perte baisse vite tant que le code est très petit, puis de moins en moins : au-delà d'une trentaine de dimensions, agrandir le code n'améliore presque plus la reconstruction. Un code de 16 à 32 dimensions est un bon compromis, soit un taux de compression de 784 / 32 = 24,5 à 784 / 16 = 49 (en nombre de valeurs : le code et les pixels sont tous des flottants).

## 7. Pour aller plus loin : un auto-encodeur épars

Un auto-encodeur épars a un code large, mais une pénalité sur ses activations n'en laisse que peu actives pour chaque image (slide « Auto-encodeurs épars »). Construisez `sparse_autoencoder` :

- l'auto-encodeur profond, avec un code de 64 unités, dont la couche ajoute à la perte une pénalité L1 de coefficient 1e-3 sur ses activations ;
- entraînez-le 10 epochs, avec le même `fit` que dans la partie 6.

Comparez-le à l'auto-encodeur à code de 64 unités de la partie 6, `autoencoders[64]` : le nombre moyen d'unités actives (> 0) du code par image de test, le nombre d'unités jamais actives, et l'erreur quadratique moyenne de reconstruction. Que coûte la parcimonie ?

**Documentation utile**

- [Les régularisations des couches Keras](https://keras.io/api/layers/regularizers/)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `layers.Dense(64, activation="relu", activity_regularizer=keras.regularizers.L1(1e-3))` : la couche du code ; Keras ajoute à la perte 1e-3 fois la somme des valeurs absolues de ses sorties
- `autoencoders[64].layers[0]` : l'encodeur de l'auto-encodeur de la partie 6 (le premier modèle de son `Sequential`)
- `codes = encoder.predict(X_test, verbose=0)`, puis `(codes > 0).sum(axis=1).mean()` : le nombre moyen d'unités actives par image ; `(codes.max(axis=0) == 0).sum()` : les unités jamais actives

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
sparse_encoder = keras.Sequential(
  [
    keras.Input(shape=(input_dim,)),
    layers.Dense(128, activation="relu"),
    layers.Dense(64, activation="relu", activity_regularizer=...),
  ]
)
sparse_decoder = ...
sparse_autoencoder = keras.Sequential([sparse_encoder, sparse_decoder])
sparse_autoencoder.compile(optimizer="adam", loss="binary_crossentropy")
sparse_autoencoder.fit(...)

for name, model in [("Sans pénalité", autoencoders[64]), ("L1", sparse_autoencoder)]:
  codes = model.layers[0].predict(X_test, verbose=0)
  error = np.mean((X_test - model.predict(X_test, verbose=0)) ** 2)
  print(f"{name} : ...")
```

### Solution

In [ ]:
sparse_encoder = keras.Sequential(
  [
    keras.Input(shape=(input_dim,)),
    layers.Dense(128, activation="relu"),
    layers.Dense(
      64, activation="relu", activity_regularizer=keras.regularizers.L1(1e-3)
    ),
  ],
  name="sparse_encoder",
)
sparse_decoder = keras.Sequential(
  [
    keras.Input(shape=(64,)),
    layers.Dense(128, activation="relu"),
    layers.Dense(input_dim, activation="sigmoid"),
  ],
  name="sparse_decoder",
)
sparse_autoencoder = keras.Sequential(
  [sparse_encoder, sparse_decoder], name="sparse_autoencoder"
)
sparse_autoencoder.compile(optimizer="adam", loss="binary_crossentropy")
sparse_autoencoder.fit(
  X_train,
  X_train,
  epochs=10,
  batch_size=256,
  validation_data=(X_test, X_test),
  verbose=0,
)

for name, model in [("Sans pénalité", autoencoders[64]), ("L1", sparse_autoencoder)]:
  codes = model.layers[0].predict(X_test, verbose=0)
  error = np.mean((X_test - model.predict(X_test, verbose=0)) ** 2)
  print(
    f"{name} : {(codes > 0).sum(axis=1).mean():.1f} unités actives par image sur 64, "
    f"{(codes.max(axis=0) == 0).sum()} jamais actives, erreur {error:.4f}"
  )

Avec la pénalité, chaque image n'active plus qu'une vingtaine d'unités du code au lieu d'une cinquantaine, et la moitié des unités ne s'activent jamais : la pénalité éteint les unités les moins utiles. La parcimonie se paie en reconstruction, dont l'erreur double presque. Le coefficient règle ce compromis : à 1e-4, l'effet est à peine visible ; plus grand, le code finit par s'éteindre. Dans un code épars, chaque unité active porte un motif qu'on peut regarder seul : c'est ce qui sert à interpréter les représentations des grands modèles de langage.